# AtencionesSalud — PySpark vs. SQL Server
Dataset: `Atenciones_atendidos_2023-dires.csv` (MINSA/DIRES San Martin), 1,048,575 filas, ~166 MB.

Este notebook usa **DataFrame**, **SparkSQL** y **RDD** (los 3 objetos; el enunciado exige al menos 2) para
ejecutar la MISMA consulta critica que ya se analizo en SQL Server (`AtencionesSalud_plan_ejecucion_indice.sql`):
total de `ATENDIDOS`/`ATENCIONES` filtrando `TIPO_SEGURO = 'S.I.S'` en enero de 2023.

**Entornos donde correr esto:**
- **Google Colab (capa gratuita):** subir el CSV a `/content/`, cambiar `CSV_PATH`, y `!pip install pyspark` en la primera celda.
- **Databricks Community Edition:** subir el CSV a DBFS (`/FileStore/tables/...`), usar `spark` (ya viene creado, no hace falta `SparkSession.builder`).
- **Docker local:** imagen `jupyter/pyspark-notebook`, montar el CSV como volumen.


In [ ]:
# Si estas en Colab, descomenta la siguiente linea:
# !pip install pyspark==3.5.3

import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

CSV_PATH = "/mnt/user-data/uploads/Atenciones_atendidos_2023-dires.csv"  # ajustar segun el entorno


## 0) Sesion de Spark
Se mide el arranque por separado: en un pipeline real este costo (JVM, planificador) se paga
una sola vez por sesion, no en cada consulta — por eso no debe mezclarse con el tiempo de la consulta.

In [ ]:
t0 = time.perf_counter()
spark = (
    SparkSession.builder
    .appName("AtencionesSalud-Analisis")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
t_startup = time.perf_counter() - t0
print(f"Arranque de SparkSession: {t_startup:.2f} s (cores asignados: {spark.sparkContext.defaultParallelism})")


## 1) DataFrame — lectura con esquema explicito
Se declara el esquema a mano (`StructType`) en vez de `inferSchema=True`: con `inferSchema` Spark
tiene que leer el archivo dos veces (una para adivinar los tipos, otra para cargar los datos).

In [ ]:
schema = StructType([
    StructField("ANIO", IntegerType()), StructField("MES", IntegerType()), StructField("DIA", IntegerType()),
    StructField("CATEGORIA", StringType()), StructField("RED", StringType()), StructField("MICRORED", StringType()),
    StructField("CODIGO_UNICO", IntegerType()), StructField("NOMBRE_ESTACLECIMIENTO", StringType()),
    StructField("PROVINCIA", StringType()), StructField("DISTRITO", StringType()),
    StructField("EDAD_REG", IntegerType()), StructField("TIPO_EDAD", StringType()), StructField("SEXO", StringType()),
    StructField("ETNIA", StringType()), StructField("TIPO_SEGURO", StringType()), StructField("UPS", StringType()),
    StructField("ATENDIDOS", IntegerType()), StructField("ATENCIONES", IntegerType()),
    StructField("FECHA_CORTE", StringType()),
])

t0 = time.perf_counter()
df = spark.read.csv(CSV_PATH, sep=";", header=True, schema=schema, encoding="UTF-8")
df = df.withColumn("Fecha", F.to_date(F.concat_ws("-", "ANIO", F.lpad("MES", 2, "0"), F.lpad("DIA", 2, "0"))))
df = df.withColumn("TIPO_SEGURO", F.trim("TIPO_SEGURO"))
df.cache()
n_rows = df.count()   # fuerza la materializacion para medir el costo real de lectura
t_read = time.perf_counter() - t0
print(f"Lectura + cache del CSV: {t_read:.2f} s ({n_rows:,} filas)")


## 2) DataFrame API — consulta critica
Total de atendidos/atenciones para `TIPO_SEGURO = 'S.I.S'` en enero de 2023 (misma consulta que en T-SQL).
Se repite 5 veces (tras el `cache()`) para medir con el dato ya en memoria, igual que se hizo en SQL Server
con `SET STATISTICS TIME ON`.

In [ ]:
def consulta_dataframe():
    return (
        df.filter((F.col("TIPO_SEGURO") == "S.I.S") &
                   (F.col("Fecha").between("2023-01-01", "2023-01-31")))
          .groupBy("TIPO_SEGURO")
          .agg(F.count("*").alias("Registros"),
               F.sum("ATENDIDOS").alias("TotalAtendidos"),
               F.sum("ATENCIONES").alias("TotalAtenciones"))
          .collect()
    )

tiempos_df = []
for _ in range(5):
    t0 = time.perf_counter()
    res_df = consulta_dataframe()
    tiempos_df.append(time.perf_counter() - t0)

print("Resultado:", res_df)
print("Tiempos (s):", [round(t, 4) for t in tiempos_df])
print(f"min={min(tiempos_df):.4f}s  promedio={sum(tiempos_df)/5:.4f}s")


## 3) SparkSQL — la misma consulta, en SQL puro
Se registra el DataFrame como vista temporal y se ejecuta con `spark.sql(...)`.

In [ ]:
df.createOrReplaceTempView("atenciones")

def consulta_sparksql():
    return spark.sql('''
        SELECT TIPO_SEGURO,
               COUNT(*) AS Registros,
               SUM(ATENDIDOS) AS TotalAtendidos,
               SUM(ATENCIONES) AS TotalAtenciones
        FROM atenciones
        WHERE TIPO_SEGURO = 'S.I.S' AND Fecha BETWEEN '2023-01-01' AND '2023-01-31'
        GROUP BY TIPO_SEGURO
    ''').collect()

tiempos_sql = []
for _ in range(5):
    t0 = time.perf_counter()
    res_sql = consulta_sparksql()
    tiempos_sql.append(time.perf_counter() - t0)

print("Resultado:", res_sql)
print("Tiempos (s):", [round(t, 4) for t in tiempos_sql])
print(f"min={min(tiempos_sql):.4f}s  promedio={sum(tiempos_sql)/5:.4f}s")


## 4) RDD — estadistica descriptiva calculada a mano
Media, desviacion estandar, minimo y maximo de `EDAD_REG` (excluyendo -1 = edad no registrada),
usando `map`/`reduce` de bajo nivel en vez de `.describe()`, para demostrar explicitamente el API de RDD.

In [ ]:
t0 = time.perf_counter()
rdd = df.select("EDAD_REG").filter(F.col("EDAD_REG") >= 0).rdd.map(lambda r: r["EDAD_REG"])
rdd.cache()
n = rdd.count()

suma = rdd.reduce(lambda a, b: a + b)
media = suma / n

suma_sq_diff = rdd.map(lambda x: (x - media) ** 2).reduce(lambda a, b: a + b)
desviacion = (suma_sq_diff / n) ** 0.5

edad_min = rdd.reduce(lambda a, b: min(a, b))
edad_max = rdd.reduce(lambda a, b: max(a, b))
t_rdd = time.perf_counter() - t0

print(f"n={n:,}  media={media:.2f}  desviacion_estandar={desviacion:.2f}  min={edad_min}  max={edad_max}")
print(f"tiempo total (count + 2 reduce + 2 reduce adicionales): {t_rdd:.2f} s")


## 5) Resumen y cierre de la sesion

In [ ]:
print("=== RESUMEN ===")
print(f"Arranque SparkSession : {t_startup:.2f} s")
print(f"Lectura CSV + cache   : {t_read:.2f} s  ({n_rows:,} filas)")
print(f"Consulta critica (DF) : min {min(tiempos_df)*1000:.1f} ms | prom {sum(tiempos_df)/5*1000:.1f} ms")
print(f"Consulta critica (SQL): min {min(tiempos_sql)*1000:.1f} ms | prom {sum(tiempos_sql)/5*1000:.1f} ms")
print(f"Stats RDD EDAD_REG    : {t_rdd*1000:.1f} ms")

spark.stop()
